# المبرمج المجهول AI — المحرك على Colab

1. **Runtime → Change runtime type → T4 GPU**
2. شغّل الخلايا بالترتيب.
3. انسخ رابط `trycloudflare.com` وضعه في لوحة التحكم ← الإعدادات ← المحرك.

In [ ]:
# 1) ربط Google Drive — البيانات والـ Checkpoints تبقى بعد انقطاع الجلسة
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ALMAJHOOL_HOME'] = '/content/drive/MyDrive/almajhool-ai'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) تثبيت المشروع
!git clone -q https://github.com/almajhool-dev/almajhool-ai /content/almajhool-ai || git -C /content/almajhool-ai pull -q
%cd /content/almajhool-ai
!pip install -q -e ".[train]"

In [ ]:
# 3) (اختياري) كلمة سر للـ API — ضعها نفسها في اللوحة
import secrets
os.environ['ALMAJHOOL_API_KEY'] = os.environ.get('ALMAJHOOL_API_KEY') or secrets.token_urlsafe(16)
print('API key:', os.environ['ALMAJHOOL_API_KEY'])

In [ ]:
# 4) تشغيل الخادم + نفق HTTPS مجاني
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time, re
server = subprocess.Popen(['python', '-m', 'almajhool', 'serve', '--port', '8000'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n✅ رابط المحرك:', m.group(0)); break

## بدون لوحة التحكم (سطر الأوامر)
ارفع ملفاتك إلى `/content/data/` ثم:

In [ ]:
!python -m almajhool add-data /content/data/* --note "دفعة من Colab"
!python -m almajhool datasets

In [ ]:
# تدريب جديد (أو أضف --parent v1 لمواصلة إصدار سابق)
!python -m almajhool train --epochs 1

In [ ]:
# استكمال تدريب انقطع:
!python -m almajhool runs
# !python -m almajhool train --resume-run run-XXXXXXXX

In [ ]:
!python -m almajhool models
!python -m almajhool backup --no-checkpoints